# Vial color and recipe-transition recovery

This notebook runs the real-data workflow in `changeover_guard.model_workflow`.
It measures full-vial color, audits recipe boundaries, plots observed recovery,
counts independent sessions, and fits only supported optical-density models.

**Available evidence:** the committed October 29 CSVs contain 27 vial labels and
six recipe changes in one session, 1,187 aligned images and three recordings.
The camera controls needed for calibration and production decisions are absent.
The shipped assessment samples 26 vials; 16 pass measurement quality checks.
These are real-data exploratory results. Controlled fixtures are used only in tests.

Read [the calibration and schema guide](../docs/transition_analysis.md). Select
the repository `.venv` kernel after `uv sync --locked --group analysis`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

_repo_root = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'pyproject.toml').exists()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError('Could not find the repository root (pyproject.toml).')

sys.path.insert(0, str(_repo_root / 'src'))
from changeover_guard.model_workflow import default_config, run_workflow

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists() and (ROOT.parent / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Run from the repository or notebooks directory.'
pd.set_option('display.max_columns', 30)
print('Repository:', ROOT)

## 1. Configure sessions and validate the clock

The default detects `data/10_29_26/vial_filled/labels/vial_recipes.csv` and
`vial_fill_frame_labels.csv`, as well as original video-5 names. Edit the video
path if more than one recording is present or it lives elsewhere.

New labels use `start_time/end_time`. The frame labels establish
`video_origin = timestamp - video_s`. For the supplied example, `2:04:48` starts
at **112.61 wall-clock elapsed seconds**, not at 7,488 seconds. This differs from
the MP4 container clock in this recording. Auto mode uses the aligned timestamped
stills. Original elapsed-second video-5 labels keep their video sampling path.

Add one `sessions` entry per independently collected recording, not per crop or
frame. Optional chart, repeatability, clean controls and calibration paths are
documented in the guide. All relative paths resolve from `ROOT`.

In [ ]:
CONFIG_FILE = ROOT / 'work' / 'transition_config.json'
CONFIG = json.loads(CONFIG_FILE.read_text(encoding='utf-8')) if CONFIG_FILE.exists() else default_config(ROOT)

# Example overrides for your local checkout:
# CONFIG['sessions'][0]['recipe_csv'] = 'data/10_29_26/vial_filled/labels/vial_recipes.csv'
# CONFIG['sessions'][0]['frame_csv'] = 'data/10_29_26/vial_filled/labels/vial_fill_frame_labels.csv'
# CONFIG['sessions'][0]['video'] = 'data/10_29_26/vial_filled/videos/your_recording.mp4'
# CONFIG['chart_csv'] = 'work/vial_chart.csv'
# CONFIG['calibration_file'] = 'outputs/transition_analysis/<run>/vial_camera_calibration.json'
# CONFIG['repeatability_csv'] = 'work/stable_vial_controls.csv'
# CONFIG['clean_controls_csv'] = 'work/verified_clean_controls.csv'

print(json.dumps(CONFIG, indent=2))

In [ ]:
result = run_workflow(CONFIG, ROOT)
report = result['report']
OUTPUT = result['output_dir']
measurements, curves, summary = result['measurements'], result['curves'], result['summary']
display(pd.Series({
    'status': report['status'], 'measured_vials': report['measured_vials'],
    'usable_vials': report['usable_vials'],
    'production_enabled': report['production_decisions_enabled'],
    'output_directory': str(OUTPUT),
}))
display(pd.DataFrame(report['label_audits']))
display(pd.Series(report['label_inventory']))
display(result['label_runs'])
display(result['label_transitions'])
if report['missing_inputs']:
    print('Missing inputs:', *report['missing_inputs'], sep='\n  ')
if report['errors']:
    display(pd.DataFrame(report['errors']))

## 2. Inspect full-vial alignment and the liquid ROI

For the new collection, each vial uses the nearest matching timestamped still
inside its final 2.5 seconds, with source time and image hash retained. Its single
still gives no temporal repeatability estimate. The MP4 runs 1,956.3 container
seconds but the image index spans 2,372 wall-clock seconds. Treating them as the
same clock would inspect the wrong vial. Original video labels use three decoded
samples near the end time. Bad samples stay in sequence and break recovery streaks.

The committed labels explicitly mark vial 10 as unobservable during a blackout;
its timing stays missing. Requested throughput after an operator error remains
unsupported until measured volumes establish it. No time window is invented.

Review the contact sheets at **every recipe change** before setting
`alignment_reviewed`. Spatial spread and adjacent-frame variation are quality
diagnostics; neither proves camera repeatability. All contact pages are saved
beside the CSV exports, including those not displayed below.

In [ ]:
if measurements.empty:
    print('No full-vial measurements: supply labels and the matching recording.')
else:
    columns = [c for c in ['session_id', 'vial', 'recipe_id', 'start_s', 'end_s',
                           'sample_times_s', 'actual_source_times_s', 'source_clock',
                           'source_clock_verified', 'source_timestamp_age_s', 'r', 'g', 'b',
                           'L', 'a', 'lab_b', 'usable', 'quality_flags', 'volume_source'] if c in measurements]
    display(measurements[columns])
for contact in sorted(OUTPUT.glob('alignment_roi_*.png'))[:4]:
    display(Image(filename=str(contact), width=1100))

## 3. Camera accuracy and stable-vial repeatability

A linear-RGB matrix needs held-out chart validation. At least five unchanged
physical controls need five independent captures each in two sessions.
Worst within-control variation and chart p95 error must both be below 2.2/3.
Alignment must be reviewed, with locked settings or a per-frame reference.

An identity transform, automatic white balance, and many adjacent production
frames cannot establish this proof. Uncertainty and production decisions remain
pending until the evidence passes for this exact calibration fingerprint.

In [ ]:
display(pd.Series({
    'camera_id': result['calibration'].camera_id,
    'calibration_id': result['calibration'].calibration_id,
    'chart_validation_delta_e': result['calibration'].validation_delta_e,
    'repeatability_proven': result['repeatability']['proven'],
    'blocking_reasons': result['repeatability']['reasons'],
}))
display(pd.DataFrame(result['repeatability']['controls']))

## 4. Clean references and observed recovery curves

Independent verified clean runs are preferred. Otherwise stable tails are marked
as **surrogates**. Vials used for a same-run tail target are withheld from that
run's evaluated curve. Surrogates never train the clean or carryover model.

Recovery needs three consecutive usable observations with ΔE00 **< 2.2**.
We record both the first vial in the streak and the later confirmation point,
in vials, mL and seconds. Insufficient follow-up or another recipe change causes
right censoring; the last observed vial is not invented as a clearance point.
Volume uses requested mL unless measured `dispensed_ml` was supplied.

In [ ]:
display(pd.DataFrame(result['targets']))
if not curves.empty:
    display(curves[['transition_id', 'vial_in_run', 'cumulative_ml',
                    'delta_e_to_clean', 'target_source', 'target_withheld', 'usable']])
for figure in sorted((OUTPUT / 'recovery_plots').glob('*.png')):
    display(Image(filename=str(figure), width=1000))

## 5. Rank observed transitions and count independent evidence

Completed and right-censored transitions are kept separate. A censored transition
may be slow, or simply insufficiently observed. The directed heatmap retains
that distinction. Never rank a censored short recording as a fast recovery.

Minimum support for evaluating one shared decay: five transitions in three
independent sessions, each with four usable vials and an independent clean
target. Counts are identifiability checks, not a statistical power calculation.
Additional frames of one experiment do not supply independent transitions.

In [ ]:
if not summary.empty:
    completed = summary[~summary.right_censored].sort_values('volume_to_clear_ml', ascending=False)
    censored = summary[summary.right_censored]
    print('Completed recoveries, longest volume first:')
    display(completed)
    print('Right-censored or low-quality transitions:')
    display(censored)
heatmap = OUTPUT / 'transition_ranking_heatmap.png'
if heatmap.exists():
    display(Image(filename=str(heatmap), width=900))
display(pd.Series({k: v for k, v in result['support'].items() if k != 'directed_pairs'}))
display(pd.DataFrame(result['support'].get('directed_pairs', [])))

## 6. Fit the clean optical-density model

Effective nonnegative T200/T400 absorption is fitted in **linear RGB relative to
independent measured T100 blanks**. Clean controls are aggregated by run and
whole recipes are held out. Tank-top Lab is provenance, not an assumed vial
absorption coefficient. Recipes outside the measured simplex are unsupported.
When controls are missing or rank deficient, the artifact contains a clear
reason and no invented coefficients.

In [ ]:
model = result['clean_model']
if model is None:
    print('Clean model:', report['clean_model_status'], report['clean_model_reason'])
else:
    display(pd.DataFrame(model.absorbance, index=['T100 blank', 'T200', 'T400'], columns=['R', 'G', 'B']))
    display(pd.Series({k: v for k, v in model.validation.items() if k != 'folds'}))
    display(pd.DataFrame(model.validation['folds']))
    # Reloadable artifact: CleanOpticalModel.from_dict(model.to_dict()).predict(recipe)

## 7. Fit the simplest supported carryover decay

The candidate is one shared exponential tau with a signed OD residual vector per
observed transition. It integrates the state over each vial's collection volume.
Whole collection sessions are held out. Their first vial initializes the
residual, and only later vials are scored against zero-carryover and constant
baselines. The candidate must improve both by 10% and have an identifiable tau.

This is a **conditional recovery forecast after the first observed vial**. It
does not claim to predict arbitrary first-vial contamination, flush/loading
effects or chemical purity. Unsupported data leaves tau null. A bootstrap tau
interval requires proven camera repeatability and at least five sessions.

In [ ]:
display(pd.Series({k: v for k, v in result['carryover'].items()
                   if k not in ['validation', 'transition_amplitudes_od']}))
display(pd.DataFrame(result['carryover']['validation']))

## 8. Production gate and exported artifacts

Decisions compare observed color to the **clean optical prediction**, never the
predicted contaminated color. Camera proof, independent clean-model validation
and interpolation support are mandatory. Only then is ΔE00 < 2.2 a pass and
ΔE00 ≥ 2.2 a fail. Until then the status is `PENDING` and uncertainty is null.

Every invocation gets its own export directory; stale successful fits cannot
appear to belong to a later incomplete run. The existing operator API is not
automatically enabled by exploratory notebook output.

In [ ]:
if not measurements.empty:
    display(measurements[['vial_key', 'recipe_id', 'production_status',
                          'production_delta_e_to_clean', 'prediction_error_p95_delta_e']])
print('Production decisions enabled:', report['production_decisions_enabled'])
print('Exports:', OUTPUT)
display(pd.DataFrame({'file': sorted(str(p.relative_to(OUTPUT)) for p in OUTPUT.iterdir() if p.is_file())}))
print('Remaining limitations:', *report['limitations'], sep='\n  ')